In [ ]:
import numpy as np
import pandas as pd
import pickle

from pathlib import Path
from tqdm import tqdm
from collections import Counter
import re
import os 
import sys
import numpy.core

sys.modules["numpy._core"] = numpy.core
sys.modules["numpy._core.multiarray"] = numpy.core.multiarray
sys.modules["numpy._core.numeric"] = numpy.core.numeric

In [3]:
PICKLE_PATH = Path(
    "/gpfs01/berens/data/data/Eyewire2/database-exports/clf_rgcs_and_acs/"
    "df_rgcs_and_acs_2026-11-02-19h.pickle"
)

df_features = pd.read_pickle(PICKLE_PATH)

In [ ]:
def split_cells(
    df_features,
    exclude_type_val=None,
    exclude_type_test=None,
    seed=42,
    split_ratio=0.15
):
    rng = np.random.default_rng(seed)

    df = df_features[
        (df_features["cellclass"] == "RGC") &
        (df_features["celltype"].notna())
    ].copy().reset_index(drop=True)

    df["cell_id"] = df["skel_file"].apply(
        lambda x: os.path.splitext(os.path.basename(x))[0]
    )

    cell_types = df["celltype"].unique().tolist()

    # Select 3 cell types for validation
    if exclude_type_val is None:
        exclude_type_val = rng.choice(
            cell_types, size=3, replace=False
        ).tolist()

    # Select 3 different cell types for test
    remaining_types = [
        t for t in cell_types if t not in exclude_type_val
    ]

    if exclude_type_test is None:
        exclude_type_test = rng.choice(
            remaining_types, size=3, replace=False
        ).tolist()

    heldout_types = exclude_type_val + exclude_type_test

    # High-confidence cells for validation and test
    high = df[df["celltype_conf"] == "High"].copy()

    val_parts = []
    test_parts = []

    # 1. Validation: held-out validation types
    val_heldout = high[
        high["celltype"].isin(exclude_type_val)
    ]

    if len(val_heldout) == 0:
        raise ValueError("No High-confidence validation cells.")

    val_parts.append(val_heldout)

    # 2. Test: held-out test types
    test_heldout = high[
        high["celltype"].isin(exclude_type_test)
    ]

    if len(test_heldout) == 0:
        raise ValueError("No High-confidence test cells.")

    test_parts.append(test_heldout)

    # 3. Remaining cell types: stratified 15% per cell type
    for cell_type in cell_types:

        if cell_type in heldout_types:
            continue

        type_data = high[high["celltype"] == cell_type]

        n = len(type_data)

        n_val = round(n * split_ratio)
        n_test = round(n * split_ratio)

        if n < n_val + n_test:
            raise ValueError(
                f"Not enough High-confidence cells for {cell_type}"
            )

        # Sample validation
        val_sample = type_data.sample(
            n=n_val,
            random_state=seed
        )

        remaining = type_data.drop(index=val_sample.index)

        # Sample test from remaining cells
        test_sample = remaining.sample(
            n=n_test,
            random_state=seed
        )

        val_parts.append(val_sample)
        test_parts.append(test_sample)

    # Combine validation and test
    val = pd.concat(val_parts)
    test = pd.concat(test_parts)

   # Training: exclude held-out types and sampled validation/test cells
    excluded_indices = val.index.union(test.index)

    train = df[
        (~df["celltype"].isin(heldout_types)) &
        (~df.index.isin(excluded_indices))
    ].copy()

    # Shuffle
    train = train.sample(frac=1, random_state=seed)
    val = val.sample(frac=1, random_state=seed)
    test = test.sample(frac=1, random_state=seed)

    return train, val, test, exclude_type_val, exclude_type_test


In [30]:
train, val, test, excluded_val, excluded_test = split_cells(df_features)

In [31]:
np.save("train_ids.npy", train["cell_id"].to_numpy(dtype=str))
np.save("val_ids.npy", val["cell_id"].to_numpy(dtype=str))
np.save("test_ids.npy", test["cell_id"].to_numpy(dtype=str))

print("Saved:")
print("Train:", len(train), "cells")
print("Validation:", len(val), "cells")
print("Test:", len(test), "cells")

Saved:
Train: 2798 cells
Validation: 428 cells
Test: 356 cells
